# ChemExpo PUC downloader

Downloads, for each of the 476 Product Use Categories (PUCs) listed at
https://comptox.epa.gov/chemexpo/pucs/, the "Products and Chemical Weight
Fractions" Excel export from its detail page
(`https://comptox.epa.gov/chemexpo/puc/{id}/`), names it
`PUC_<Gen Cat>_<Prod Fam>_<Prod Type>.xlsx` (segments that are blank for a
given PUC are omitted), sorts it into a local folder per Gen Cat (35 of
them), then uploads the whole tree into a Google Drive folder you choose.

**Before running:**
1. Runtime -> Change runtime type -> keep it as the default (no GPU needed).
2. Have the target Google Drive folder's ID handy. From a share link like
   `https://drive.google.com/drive/folders/1tDncfoHC14dWet34H34SEukt1BLi4aPT?usp=sharing`
   the ID is the part after `/folders/` and before `?`, i.e.
   `1tDncfoHC14dWet34H34SEukt1BLi4aPT`. Paste it into `DRIVE_FOLDER_ID` in
   the upload cell near the end. You (or whoever runs this) need at least
   Editor access on that folder, which the notebook will ask you to grant
   via a Google sign-in popup when it reaches the upload step.
3. This is NOT tested against the live site from where I generated it
   (this sandbox's network policy blocks comptox.epa.gov entirely, so I
   could not click through and confirm the download button's exact
   behavior). The PUC list itself (476 rows, Gen Cat/Prod Fam/Prod
   Type/id) IS real -- extracted directly from the PUCs listing page you
   sent me, not guessed. **The one thing most likely to need a small fix
   is the download-button locator in the "Download loop" cell** if
   ChemExpo's button text or markup differs from what's assumed
   (`get_by_text("Download Products and Chemical Weight Fractions")`) --
   if the first few PUCs fail with a timeout, open a PUC detail page,
   right-click the button, "Inspect", and adjust the locator to match.
   The loop logs every failure to `failed_pucs.csv` instead of stopping,
   so a bad locator shows up immediately as "476 failed" rather than
   silently wasting the run.


## 1. Install Playwright and the Google Drive API client

In [ ]:
!pip install -q playwright google-api-python-client google-auth-httplib2 google-auth-oauthlib
!playwright install chromium
!playwright install-deps chromium


## 2. The 476 PUCs (Gen Cat / Prod Fam / Prod Type / id)

Extracted from the `<script id="tabledata">` JSON embedded in the PUCs listing page (https://comptox.epa.gov/chemexpo/pucs/) -- the whole table loads server-side as one JSON blob rather than paging via API calls, so this is the exact, complete list rather than something reconstructed from a partial page scrape.

In [ ]:
import json

PUC_LIST = json.loads(r'''[{"id":316,"gen_cat":"Batteries","prod_fam":"","prod_type":""},{"id":450,"gen_cat":"Batteries","prod_fam":"electronic device","prod_type":""},{"id":443,"gen_cat":"Batteries","prod_fam":"electronic device","prod_type":"camera"},{"id":441,"gen_cat":"Batteries","prod_fam":"electronic device","prod_type":"laptop"},{"id":440,"gen_cat":"Batteries","prod_fam":"electronic device","prod_type":"phone"},{"id":436,"gen_cat":"Batteries","prod_fam":"general use","prod_type":""},{"id":437,"gen_cat":"Batteries","prod_fam":"general use","prod_type":"alkaline"},{"id":438,"gen_cat":"Batteries","prod_fam":"general use","prod_type":"lithium"},{"id":439,"gen_cat":"Batteries","prod_fam":"general use","prod_type":"rechargeable"},{"id":442,"gen_cat":"Batteries","prod_fam":"vehicle","prod_type":""},{"id":444,"gen_cat":"Batteries","prod_fam":"watch","prod_type":""},{"id":305,"gen_cat":"Cons. electronics, mech. appliances, and machinery","prod_fam":"","prod_type":""},{"id":429,"gen_cat":"Cons. electronics, mech. appliances, and machinery","prod_fam":"light bulbs","prod_type":""},{"id":309,"gen_cat":"Construction and building materials","prod_fam":"","prod_type":""},{"id":493,"gen_cat":"Construction and building materials","prod_fam":"flooring","prod_type":""},{"id":445,"gen_cat":"Construction and building materials","prod_fam":"insulation","prod_type":""},{"id":495,"gen_cat":"Construction and building materials","prod_fam":"roofing","prod_type":""},{"id":497,"gen_cat":"Construction and building materials","prod_fam":"tiling","prod_type":""},{"id":496,"gen_cat":"Construction and building materials","prod_fam":"waterproofing materials","prod_type":""},{"id":312,"gen_cat":"Food contact items","prod_fam":"","prod_type":""},{"id":313,"gen_cat":"Furniture and furnishings","prod_fam":"","prod_type":""},{"id":306,"gen_cat":"Industrial machinery","prod_fam":"","prod_type":""},{"id":402,"gen_cat":"Manufacturing Components","prod_fam":"","prod_type":""},{"id":314,"gen_cat":"Other direct contact consumer goods","prod_fam":"","prod_type":""},{"id":426,"gen_cat":"Other direct contact consumer goods","prod_fam":"apparel","prod_type":""},{"id":409,"gen_cat":"Other direct contact consumer goods","prod_fam":"gun ammo","prod_type":""},{"id":468,"gen_cat":"Other direct contact consumer goods","prod_fam":"medical products","prod_type":""},{"id":315,"gen_cat":"Other indirect contact consumer goods","prod_fam":"","prod_type":""},{"id":308,"gen_cat":"Other vehicles/mass transit","prod_fam":"","prod_type":""},{"id":311,"gen_cat":"Packaging (non-food contact)","prod_fam":"","prod_type":""},{"id":307,"gen_cat":"Road vehicles","prod_fam":"","prod_type":""},{"id":352,"gen_cat":"Tools","prod_fam":"","prod_type":""},{"id":310,"gen_cat":"Toys and children's products","prod_fam":"","prod_type":""},{"id":20,"gen_cat":"Arts and crafts/office supplies","prod_fam":"","prod_type":""},{"id":1,"gen_cat":"Arts and crafts/office supplies","prod_fam":"body paint","prod_type":""},{"id":322,"gen_cat":"Arts and crafts/office supplies","prod_fam":"body paint","prod_type":"tattoo ink"},{"id":6,"gen_cat":"Arts and crafts/office supplies","prod_fam":"children's arts and crafts","prod_type":""},{"id":2,"gen_cat":"Arts and crafts/office supplies","prod_fam":"children's arts and crafts","prod_type":"bubble solution"},{"id":359,"gen_cat":"Arts and crafts/office supplies","prod_fam":"children's arts and crafts","prod_type":"chalk"},{"id":3,"gen_cat":"Arts and crafts/office supplies","prod_fam":"children's arts and crafts","prod_type":"crayons"},{"id":4,"gen_cat":"Arts and crafts/office supplies","prod_fam":"children's arts and crafts","prod_type":"finger paint"},{"id":471,"gen_cat":"Arts and crafts/office supplies","prod_fam":"children's arts and crafts","prod_type":"glowsticks"},{"id":5,"gen_cat":"Arts and crafts/office supplies","prod_fam":"children's arts and crafts","prod_type":"modeling clay"},{"id":335,"gen_cat":"Arts and crafts/office supplies","prod_fam":"children's arts and crafts","prod_type":"slime"},{"id":9,"gen_cat":"Arts and crafts/office supplies","prod_fam":"fabric treatment and dye","prod_type":""},{"id":7,"gen_cat":"Arts and crafts/office supplies","prod_fam":"fabric treatment and dye","prod_type":"fabric dye"},{"id":8,"gen_cat":"Arts and crafts/office supplies","prod_fam":"fabric treatment and dye","prod_type":"fabric paints and sealers"},{"id":489,"gen_cat":"Arts and crafts/office supplies","prod_fam":"fine art supplies","prod_type":""},{"id":491,"gen_cat":"Arts and crafts/office supplies","prod_fam":"fine art supplies","prod_type":"inks"},{"id":490,"gen_cat":"Arts and crafts/office supplies","prod_fam":"fine art supplies","prod_type":"modeling clay"},{"id":10,"gen_cat":"Arts and crafts/office supplies","prod_fam":"fog machine","prod_type":""},{"id":17,"gen_cat":"Arts and crafts/office supplies","prod_fam":"general arts and crafts supplies","prod_type":""},{"id":11,"gen_cat":"Arts and crafts/office supplies","prod_fam":"general arts and crafts supplies","prod_type":"arts and crafts adhesive"},{"id":12,"gen_cat":"Arts and crafts/office supplies","prod_fam":"general arts and crafts supplies","prod_type":"arts and crafts cleaner"},{"id":13,"gen_cat":"Arts and crafts/office supplies","prod_fam":"general arts and crafts supplies","prod_type":"arts and crafts finish"},{"id":14,"gen_cat":"Arts and crafts/office supplies","prod_fam":"general arts and crafts supplies","prod_type":"arts and crafts paint"},{"id":15,"gen_cat":"Arts and crafts/office supplies","prod_fam":"general arts and crafts supplies","prod_type":"craft kit"},{"id":16,"gen_cat":"Arts and crafts/office supplies","prod_fam":"general arts and crafts supplies","prod_type":"flocking"},{"id":18,"gen_cat":"Arts and crafts/office supplies","prod_fam":"home office","prod_type":""},{"id":469,"gen_cat":"Arts and crafts/office supplies","prod_fam":"home office","prod_type":"pencils"},{"id":22,"gen_cat":"Arts and crafts/office supplies","prod_fam":"home office","prod_type":"pens and markers"},{"id":23,"gen_cat":"Arts and crafts/office supplies","prod_fam":"home office","prod_type":"white out"},{"id":19,"gen_cat":"Arts and crafts/office supplies","prod_fam":"pottery making","prod_type":""},{"id":24,"gen_cat":"Arts and crafts/office supplies","prod_fam":"pottery making","prod_type":"glaze"},{"id":48,"gen_cat":"Cleaning products and household care","prod_fam":"","prod_type":""},{"id":25,"gen_cat":"Cleaning products and household care","prod_fam":"air freshener","prod_type":""},{"id":44,"gen_cat":"Cleaning products and household care","prod_fam":"appliance cleaner","prod_type":""},{"id":463,"gen_cat":"Cleaning products and household care","prod_fam":"appliance cleaner","prod_type":"dishwasher cleaner"},{"id":68,"gen_cat":"Cleaning products and household care","prod_fam":"appliance cleaner","prod_type":"oven cleaner"},{"id":425,"gen_cat":"Cleaning products and household care","prod_fam":"appliance cleaner","prod_type":"washing machine cleaner"},{"id":27,"gen_cat":"Cleaning products and household care","prod_fam":"bathroom","prod_type":""},{"id":26,"gen_cat":"Cleaning products and household care","prod_fam":"bathroom","prod_type":"bathroom cleaner"},{"id":32,"gen_cat":"Cleaning products and household care","prod_fam":"carpet and floor","prod_type":""},{"id":28,"gen_cat":"Cleaning products and household care","prod_fam":"carpet and floor","prod_type":"carpet cleaner"},{"id":29,"gen_cat":"Cleaning products and household care","prod_fam":"carpet and floor","prod_type":"carpet deodorizer"},{"id":30,"gen_cat":"Cleaning products and household care","prod_fam":"carpet and floor","prod_type":"floor cleaner"},{"id":31,"gen_cat":"Cleaning products and household care","prod_fam":"carpet and floor","prod_type":"floor polish"},{"id":36,"gen_cat":"Cleaning products and household care","prod_fam":"dishwasher and dishes","prod_type":""},{"id":33,"gen_cat":"Cleaning products and household care","prod_fam":"dishwasher and dishes","prod_type":"automatic dishwashing additive"},{"id":34,"gen_cat":"Cleaning products and household care","prod_fam":"dishwasher and dishes","prod_type":"automatic dishwashing detergent"},{"id":35,"gen_cat":"Cleaning products and household care","prod_fam":"dishwasher and dishes","prod_type":"dish soap"},{"id":37,"gen_cat":"Cleaning products and household care","prod_fam":"drain products","prod_type":""},{"id":49,"gen_cat":"Cleaning products and household care","prod_fam":"fireplace","prod_type":""},{"id":39,"gen_cat":"Cleaning products and household care","prod_fam":"general household cleaning","prod_type":""},{"id":50,"gen_cat":"Cleaning products and household care","prod_fam":"general household cleaning","prod_type":"bleach"},{"id":51,"gen_cat":"Cleaning products and household care","prod_fam":"general household cleaning","prod_type":"disinfectant"},{"id":52,"gen_cat":"Cleaning products and household care","prod_fam":"general household cleaning","prod_type":"glass cleaner"},{"id":54,"gen_cat":"Cleaning products and household care","prod_fam":"general household cleaning","prod_type":"heavy duty cleaner"},{"id":38,"gen_cat":"Cleaning products and household care","prod_fam":"general household cleaning","prod_type":"surface cleaner"},{"id":40,"gen_cat":"Cleaning products and household care","prod_fam":"hand cleaner","prod_type":""},{"id":55,"gen_cat":"Cleaning products and household care","prod_fam":"houseplant care","prod_type":""},{"id":393,"gen_cat":"Cleaning products and household care","prod_fam":"jewelry","prod_type":""},{"id":303,"gen_cat":"Cleaning products and household care","prod_fam":"jewelry","prod_type":"jewelry cleaner"},{"id":56,"gen_cat":"Cleaning products and household care","prod_fam":"lamp oil/lighter fluid","prod_type":""},{"id":42,"gen_cat":"Cleaning products and household care","prod_fam":"laundry and fabric treatment","prod_type":""},{"id":41,"gen_cat":"Cleaning products and household care","prod_fam":"laundry and fabric treatment","prod_type":"anti-static spray"},{"id":57,"gen_cat":"Cleaning products and household care","prod_fam":"laundry and fabric treatment","prod_type":"dry cleaner"},{"id":58,"gen_cat":"Cleaning products and household care","prod_fam":"laundry and fabric treatment","prod_type":"dryer sheets"},{"id":59,"gen_cat":"Cleaning products and household care","prod_fam":"laundry and fabric treatment","prod_type":"fabric deodorizer"},{"id":60,"gen_cat":"Cleaning products and household care","prod_fam":"laundry and fabric treatment","prod_type":"fabric protectant"},{"id":61,"gen_cat":"Cleaning products and household care","prod_fam":"laundry and fabric treatment","prod_type":"fabric softener"},{"id":62,"gen_cat":"Cleaning products and household care","prod_fam":"laundry and fabric treatment","prod_type":"laundry detergent"},{"id":63,"gen_cat":"Cleaning products and household care","prod_fam":"laundry and fabric treatment","prod_type":"laundry fragrance"},{"id":64,"gen_cat":"Cleaning products and household care","prod_fam":"laundry and fabric treatment","prod_type":"laundry stain remover"},{"id":65,"gen_cat":"Cleaning products and household care","prod_fam":"laundry and fabric treatment","prod_type":"laundry starch"},{"id":66,"gen_cat":"Cleaning products and household care","prod_fam":"lime remover","prod_type":""},{"id":43,"gen_cat":"Cleaning products and household care","prod_fam":"metal specific","prod_type":""},{"id":460,"gen_cat":"Cleaning products and household care","prod_fam":"metal specific","prod_type":"metal cleaner"},{"id":67,"gen_cat":"Cleaning products and household care","prod_fam":"metal specific","prod_type":"metal polish"},{"id":45,"gen_cat":"Cleaning products and household care","prod_fam":"shoes","prod_type":""},{"id":69,"gen_cat":"Cleaning products and household care","prod_fam":"shoes","prod_type":"shoe polish or protectant"},{"id":46,"gen_cat":"Cleaning products and household care","prod_fam":"upholstery specific","prod_type":""},{"id":70,"gen_cat":"Cleaning products and household care","prod_fam":"upholstery specific","prod_type":"upholstery cleaner"},{"id":47,"gen_cat":"Cleaning products and household care","prod_fam":"wood specific","prod_type":""},{"id":461,"gen_cat":"Cleaning products and household care","prod_fam":"wood specific","prod_type":"wood cleaner"},{"id":71,"gen_cat":"Cleaning products and household care","prod_fam":"wood specific","prod_type":"wood polish"},{"id":72,"gen_cat":"Electronics/small appliances","prod_fam":"","prod_type":""},{"id":479,"gen_cat":"Electronics/small appliances","prod_fam":"3D printing","prod_type":""},{"id":480,"gen_cat":"Electronics/small appliances","prod_fam":"3D printing","prod_type":"3D printing filament"},{"id":75,"gen_cat":"Electronics/small appliances","prod_fam":"computers and accessories/supplies","prod_type":""},{"id":73,"gen_cat":"Electronics/small appliances","prod_fam":"computers and accessories/supplies","prod_type":"printer ink"},{"id":74,"gen_cat":"Electronics/small appliances","prod_fam":"computers and accessories/supplies","prod_type":"printer toner"},{"id":76,"gen_cat":"Electronics/small appliances","prod_fam":"electronics cleaner","prod_type":""},{"id":344,"gen_cat":"Food and drug","prod_fam":"","prod_type":""},{"id":345,"gen_cat":"Food and drug","prod_fam":"food products","prod_type":""},{"id":454,"gen_cat":"Food and drug","prod_fam":"food products","prod_type":"infant formula and baby food"},{"id":347,"gen_cat":"Food and drug","prod_fam":"pharmaceuticals","prod_type":""},{"id":431,"gen_cat":"Food and drug","prod_fam":"pharmaceuticals","prod_type":"children's"},{"id":419,"gen_cat":"Food and drug","prod_fam":"pharmaceuticals","prod_type":"over the counter"},{"id":411,"gen_cat":"Food and drug","prod_fam":"pharmaceuticals","prod_type":"vaccines"},{"id":388,"gen_cat":"Food and drug","prod_fam":"smoking-related products","prod_type":""},{"id":350,"gen_cat":"Food and drug","prod_fam":"smoking-related products","prod_type":"smoking cessation products"},{"id":348,"gen_cat":"Food and drug","prod_fam":"smoking-related products","prod_type":"tobacco products"},{"id":349,"gen_cat":"Food and drug","prod_fam":"smoking-related products","prod_type":"vaping chemicals"},{"id":346,"gen_cat":"Food and drug","prod_fam":"supplements","prod_type":""},{"id":77,"gen_cat":"Home maintenance","prod_fam":"","prod_type":""},{"id":82,"gen_cat":"Home maintenance","prod_fam":"adhesives and adhesive removers","prod_type":""},{"id":78,"gen_cat":"Home maintenance","prod_fam":"adhesives and adhesive removers","prod_type":"adhesive remover"},{"id":79,"gen_cat":"Home maintenance","prod_fam":"adhesives and adhesive removers","prod_type":"multipurpose adhesive"},{"id":81,"gen_cat":"Home maintenance","prod_fam":"adhesives and adhesive removers","prod_type":"wood adhesive"},{"id":84,"gen_cat":"Home maintenance","prod_fam":"caulk/sealant","prod_type":""},{"id":85,"gen_cat":"Home maintenance","prod_fam":"concrete","prod_type":""},{"id":354,"gen_cat":"Home maintenance","prod_fam":"concrete","prod_type":"concrete paint"},{"id":86,"gen_cat":"Home maintenance","prod_fam":"corrosion protection","prod_type":""},{"id":87,"gen_cat":"Home maintenance","prod_fam":"degreaser","prod_type":""},{"id":88,"gen_cat":"Home maintenance","prod_fam":"finish","prod_type":""},{"id":296,"gen_cat":"Home maintenance","prod_fam":"finish","prod_type":"wood finish"},{"id":90,"gen_cat":"Home maintenance","prod_fam":"insulation","prod_type":""},{"id":89,"gen_cat":"Home maintenance","prod_fam":"insulation","prod_type":"spray foam"},{"id":91,"gen_cat":"Home maintenance","prod_fam":"lock deicer","prod_type":""},{"id":92,"gen_cat":"Home maintenance","prod_fam":"lubricant","prod_type":""},{"id":93,"gen_cat":"Home maintenance","prod_fam":"paint/stain and related products","prod_type":""},{"id":94,"gen_cat":"Home maintenance","prod_fam":"paint/stain and related products","prod_type":"oil-based paint"},{"id":95,"gen_cat":"Home maintenance","prod_fam":"paint/stain and related products","prod_type":"paint"},{"id":97,"gen_cat":"Home maintenance","prod_fam":"paint/stain and related products","prod_type":"paint cleaner"},{"id":98,"gen_cat":"Home maintenance","prod_fam":"paint/stain and related products","prod_type":"paint texture"},{"id":99,"gen_cat":"Home maintenance","prod_fam":"paint/stain and related products","prod_type":"paint thinner"},{"id":100,"gen_cat":"Home maintenance","prod_fam":"paint/stain and related products","prod_type":"primer"},{"id":101,"gen_cat":"Home maintenance","prod_fam":"paint/stain and related products","prod_type":"stain"},{"id":102,"gen_cat":"Home maintenance","prod_fam":"paint/stain and related products","prod_type":"stripper"},{"id":96,"gen_cat":"Home maintenance","prod_fam":"paint/stain and related products","prod_type":"water-based paint"},{"id":103,"gen_cat":"Home maintenance","prod_fam":"patch and repair","prod_type":""},{"id":105,"gen_cat":"Home maintenance","prod_fam":"patch and repair","prod_type":"putty or filler"},{"id":104,"gen_cat":"Home maintenance","prod_fam":"patch and repair","prod_type":"wall patch and repair"},{"id":106,"gen_cat":"Home maintenance","prod_fam":"plumbing","prod_type":""},{"id":108,"gen_cat":"Home maintenance","prod_fam":"roof","prod_type":""},{"id":109,"gen_cat":"Home maintenance","prod_fam":"septic system","prod_type":""},{"id":110,"gen_cat":"Home maintenance","prod_fam":"surface sealers","prod_type":""},{"id":111,"gen_cat":"Home maintenance","prod_fam":"surface sealers","prod_type":"glass surface sealer"},{"id":112,"gen_cat":"Home maintenance","prod_fam":"surface sealers","prod_type":"stone surface sealer"},{"id":113,"gen_cat":"Home maintenance","prod_fam":"tiling","prod_type":""},{"id":114,"gen_cat":"Home maintenance","prod_fam":"tiling","prod_type":"grout sealer"},{"id":115,"gen_cat":"Home maintenance","prod_fam":"tiling","prod_type":"mortar or grout"},{"id":117,"gen_cat":"Landscape/yard","prod_fam":"","prod_type":""},{"id":119,"gen_cat":"Landscape/yard","prod_fam":"grill/camping fuel","prod_type":""},{"id":120,"gen_cat":"Landscape/yard","prod_fam":"herbicide","prod_type":""},{"id":121,"gen_cat":"Landscape/yard","prod_fam":"lawn fertilizer","prod_type":""},{"id":122,"gen_cat":"Landscape/yard","prod_fam":"lawnmower","prod_type":""},{"id":123,"gen_cat":"Landscape/yard","prod_fam":"lawnmower","prod_type":"lawnmower fluids"},{"id":118,"gen_cat":"Landscape/yard","prod_fam":"outdoor cleaner","prod_type":""},{"id":124,"gen_cat":"Landscape/yard","prod_fam":"plants and garden","prod_type":""},{"id":125,"gen_cat":"Landscape/yard","prod_fam":"plants and garden","prod_type":"garden care"},{"id":126,"gen_cat":"Landscape/yard","prod_fam":"plants and garden","prod_type":"garden fertilizer"},{"id":127,"gen_cat":"Landscape/yard","prod_fam":"plants and garden","prod_type":"mulch"},{"id":128,"gen_cat":"Landscape/yard","prod_fam":"plants and garden","prod_type":"potting soil"},{"id":129,"gen_cat":"Landscape/yard","prod_fam":"plants and garden","prod_type":"tree care products"},{"id":131,"gen_cat":"Landscape/yard","prod_fam":"pool chemicals","prod_type":""},{"id":133,"gen_cat":"Landscape/yard","prod_fam":"pool chemicals","prod_type":"chlorine and bromine"},{"id":134,"gen_cat":"Landscape/yard","prod_fam":"pool chemicals","prod_type":"pH control"},{"id":132,"gen_cat":"Landscape/yard","prod_fam":"pool chemicals","prod_type":"pool algaecide"},{"id":135,"gen_cat":"Landscape/yard","prod_fam":"pool chemicals","prod_type":"shock"},{"id":136,"gen_cat":"Landscape/yard","prod_fam":"surface deicer","prod_type":""},{"id":389,"gen_cat":"Other consumer products","prod_fam":"","prod_type":""},{"id":478,"gen_cat":"Other consumer products","prod_fam":"fireworks","prod_type":""},{"id":390,"gen_cat":"Other consumer products","prod_fam":"personal safety","prod_type":""},{"id":334,"gen_cat":"Other consumer products","prod_fam":"personal safety","prod_type":"pepper spray"},{"id":137,"gen_cat":"Personal care","prod_fam":"","prod_type":""},{"id":139,"gen_cat":"Personal care","prod_fam":"acne treatment","prod_type":""},{"id":138,"gen_cat":"Personal care","prod_fam":"acne treatment","prod_type":"acne spot treatment"},{"id":140,"gen_cat":"Personal care","prod_fam":"acne treatment","prod_type":"face scrub"},{"id":141,"gen_cat":"Personal care","prod_fam":"acne treatment","prod_type":"face wash"},{"id":328,"gen_cat":"Personal care","prod_fam":"after sun product","prod_type":""},{"id":392,"gen_cat":"Personal care","prod_fam":"body","prod_type":""},{"id":143,"gen_cat":"Personal care","prod_fam":"body","prod_type":"body adhesive"},{"id":329,"gen_cat":"Personal care","prod_fam":"body","prod_type":"body firming lotion"},{"id":153,"gen_cat":"Personal care","prod_fam":"body","prod_type":"body oil"},{"id":154,"gen_cat":"Personal care","prod_fam":"body","prod_type":"body powder"},{"id":144,"gen_cat":"Personal care","prod_fam":"body care set","prod_type":""},{"id":145,"gen_cat":"Personal care","prod_fam":"body hygiene","prod_type":""},{"id":146,"gen_cat":"Personal care","prod_fam":"body hygiene","prod_type":"bar soap"},{"id":147,"gen_cat":"Personal care","prod_fam":"body hygiene","prod_type":"body scrub"},{"id":148,"gen_cat":"Personal care","prod_fam":"body hygiene","prod_type":"body wash"},{"id":149,"gen_cat":"Personal care","prod_fam":"body hygiene","prod_type":"body wipes"},{"id":304,"gen_cat":"Personal care","prod_fam":"body hygiene","prod_type":"feminine hygiene"},{"id":150,"gen_cat":"Personal care","prod_fam":"body hygiene","prod_type":"hand sanitizer"},{"id":151,"gen_cat":"Personal care","prod_fam":"body hygiene","prod_type":"hand soap"},{"id":152,"gen_cat":"Personal care","prod_fam":"body hygiene","prod_type":"hand wipes"},{"id":155,"gen_cat":"Personal care","prod_fam":"child specific","prod_type":""},{"id":415,"gen_cat":"Personal care","prod_fam":"child specific","prod_type":"baby dental care"},{"id":156,"gen_cat":"Personal care","prod_fam":"child specific","prod_type":"baby lotion"},{"id":157,"gen_cat":"Personal care","prod_fam":"child specific","prod_type":"baby oil"},{"id":158,"gen_cat":"Personal care","prod_fam":"child specific","prod_type":"baby powder"},{"id":159,"gen_cat":"Personal care","prod_fam":"child specific","prod_type":"baby shampoo"},{"id":160,"gen_cat":"Personal care","prod_fam":"child specific","prod_type":"baby wash"},{"id":161,"gen_cat":"Personal care","prod_fam":"child specific","prod_type":"baby wipes"},{"id":162,"gen_cat":"Personal care","prod_fam":"child specific","prod_type":"bath paints/crayons"},{"id":163,"gen_cat":"Personal care","prod_fam":"child specific","prod_type":"diaper cream"},{"id":170,"gen_cat":"Personal care","prod_fam":"deodorant","prod_type":""},{"id":172,"gen_cat":"Personal care","prod_fam":"eye care and contacts","prod_type":""},{"id":171,"gen_cat":"Personal care","prod_fam":"eye care and contacts","prod_type":"contact care"},{"id":173,"gen_cat":"Personal care","prod_fam":"eye care and contacts","prod_type":"eye cream"},{"id":174,"gen_cat":"Personal care","prod_fam":"eye care and contacts","prod_type":"eye drops"},{"id":298,"gen_cat":"Personal care","prod_fam":"eye care and contacts","prod_type":"eye products"},{"id":175,"gen_cat":"Personal care","prod_fam":"eye care and contacts","prod_type":"eyelid spray"},{"id":177,"gen_cat":"Personal care","prod_fam":"facial cleansing and moisturizing","prod_type":""},{"id":178,"gen_cat":"Personal care","prod_fam":"facial cleansing and moisturizing","prod_type":"face cleansing wipes"},{"id":179,"gen_cat":"Personal care","prod_fam":"facial cleansing and moisturizing","prod_type":"face cream/moisturizer"},{"id":180,"gen_cat":"Personal care","prod_fam":"facial cleansing and moisturizing","prod_type":"face mask"},{"id":181,"gen_cat":"Personal care","prod_fam":"facial cleansing and moisturizing","prod_type":"face scrub"},{"id":182,"gen_cat":"Personal care","prod_fam":"facial cleansing and moisturizing","prod_type":"face wash"},{"id":430,"gen_cat":"Personal care","prod_fam":"facial cleansing and moisturizing","prod_type":"facial hair conditioning"},{"id":183,"gen_cat":"Personal care","prod_fam":"foot care","prod_type":""},{"id":184,"gen_cat":"Personal care","prod_fam":"fragrance","prod_type":""},{"id":456,"gen_cat":"Personal care","prod_fam":"fragrance","prod_type":"body mist"},{"id":457,"gen_cat":"Personal care","prod_fam":"fragrance","prod_type":"body spray"},{"id":462,"gen_cat":"Personal care","prod_fam":"fragrance","prod_type":"essential oils"},{"id":186,"gen_cat":"Personal care","prod_fam":"general moisturizing","prod_type":""},{"id":185,"gen_cat":"Personal care","prod_fam":"general moisturizing","prod_type":"hand/body lotion"},{"id":187,"gen_cat":"Personal care","prod_fam":"glitter","prod_type":""},{"id":188,"gen_cat":"Personal care","prod_fam":"hair coloring","prod_type":""},{"id":362,"gen_cat":"Personal care","prod_fam":"hair coloring","prod_type":"beard and mustache"},{"id":189,"gen_cat":"Personal care","prod_fam":"hair coloring","prod_type":"hair bleach"},{"id":190,"gen_cat":"Personal care","prod_fam":"hair coloring","prod_type":"hair color - permanent"},{"id":191,"gen_cat":"Personal care","prod_fam":"hair coloring","prod_type":"hair color - professional"},{"id":192,"gen_cat":"Personal care","prod_fam":"hair coloring","prod_type":"hair color - temporary"},{"id":193,"gen_cat":"Personal care","prod_fam":"hair coloring","prod_type":"hair color activator"},{"id":194,"gen_cat":"Personal care","prod_fam":"hair coloring","prod_type":"hair color developer"},{"id":195,"gen_cat":"Personal care","prod_fam":"hair coloring","prod_type":"hair color toner"},{"id":197,"gen_cat":"Personal care","prod_fam":"hair styling and care","prod_type":""},{"id":196,"gen_cat":"Personal care","prod_fam":"hair styling and care","prod_type":"dry shampoo"},{"id":198,"gen_cat":"Personal care","prod_fam":"hair styling and care","prod_type":"ethnic hair care"},{"id":297,"gen_cat":"Personal care","prod_fam":"hair styling and care","prod_type":"hair conditioner"},{"id":200,"gen_cat":"Personal care","prod_fam":"hair styling and care","prod_type":"hair conditioner - leave-in"},{"id":201,"gen_cat":"Personal care","prod_fam":"hair styling and care","prod_type":"hair conditioning treatment"},{"id":202,"gen_cat":"Personal care","prod_fam":"hair styling and care","prod_type":"hair conditioning treatment - professional"},{"id":203,"gen_cat":"Personal care","prod_fam":"hair styling and care","prod_type":"hair relaxer"},{"id":204,"gen_cat":"Personal care","prod_fam":"hair styling and care","prod_type":"hair spray"},{"id":205,"gen_cat":"Personal care","prod_fam":"hair styling and care","prod_type":"hair styling"},{"id":208,"gen_cat":"Personal care","prod_fam":"hair styling and care","prod_type":"lice shampoo"},{"id":209,"gen_cat":"Personal care","prod_fam":"hair styling and care","prod_type":"scalp treatment"},{"id":210,"gen_cat":"Personal care","prod_fam":"hair styling and care","prod_type":"shampoo"},{"id":211,"gen_cat":"Personal care","prod_fam":"hair styling and care","prod_type":"shampoo - dandruff"},{"id":212,"gen_cat":"Personal care","prod_fam":"liniment","prod_type":""},{"id":213,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":""},{"id":214,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"blush/bronzer"},{"id":327,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"combination makeup products"},{"id":215,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"cosmetic tool cleaner"},{"id":216,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"eye liner"},{"id":217,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"eye makeup"},{"id":361,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"eye primer"},{"id":218,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"eye shadow"},{"id":325,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"eyebrow makeup"},{"id":219,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"face powder"},{"id":220,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"foundation/concealer"},{"id":221,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"lip balm"},{"id":222,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"lip color"},{"id":223,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"lip gloss"},{"id":224,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"lip liner"},{"id":326,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"lip plumper"},{"id":225,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"makeup primer"},{"id":226,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"makeup remover"},{"id":227,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"makeup set"},{"id":228,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"mascara"},{"id":229,"gen_cat":"Personal care","prod_fam":"makeup and related","prod_type":"toner"},{"id":230,"gen_cat":"Personal care","prod_fam":"nails","prod_type":""},{"id":231,"gen_cat":"Personal care","prod_fam":"nails","prod_type":"nail adhesive"},{"id":232,"gen_cat":"Personal care","prod_fam":"nails","prod_type":"nail polish"},{"id":233,"gen_cat":"Personal care","prod_fam":"nails","prod_type":"nail polish remover"},{"id":342,"gen_cat":"Personal care","prod_fam":"nails","prod_type":"nail treatment"},{"id":164,"gen_cat":"Personal care","prod_fam":"oral and dental care","prod_type":""},{"id":424,"gen_cat":"Personal care","prod_fam":"oral and dental care","prod_type":"breath freshener"},{"id":165,"gen_cat":"Personal care","prod_fam":"oral and dental care","prod_type":"denture adhesive"},{"id":166,"gen_cat":"Personal care","prod_fam":"oral and dental care","prod_type":"denture cleaner"},{"id":167,"gen_cat":"Personal care","prod_fam":"oral and dental care","prod_type":"mouthwash"},{"id":455,"gen_cat":"Personal care","prod_fam":"oral and dental care","prod_type":"pain relief"},{"id":168,"gen_cat":"Personal care","prod_fam":"oral and dental care","prod_type":"teeth whitener"},{"id":169,"gen_cat":"Personal care","prod_fam":"oral and dental care","prod_type":"toothpaste"},{"id":449,"gen_cat":"Personal care","prod_fam":"ostomy products","prod_type":""},{"id":234,"gen_cat":"Personal care","prod_fam":"self-tanner","prod_type":""},{"id":235,"gen_cat":"Personal care","prod_fam":"sexual wellness","prod_type":""},{"id":236,"gen_cat":"Personal care","prod_fam":"shaving and hair removal","prod_type":""},{"id":237,"gen_cat":"Personal care","prod_fam":"shaving and hair removal","prod_type":"aftershave"},{"id":238,"gen_cat":"Personal care","prod_fam":"shaving and hair removal","prod_type":"clipper lubricant/cleaner"},{"id":239,"gen_cat":"Personal care","prod_fam":"shaving and hair removal","prod_type":"depilatory"},{"id":240,"gen_cat":"Personal care","prod_fam":"shaving and hair removal","prod_type":"shaving cream"},{"id":241,"gen_cat":"Personal care","prod_fam":"shaving and hair removal","prod_type":"waxing"},{"id":330,"gen_cat":"Personal care","prod_fam":"skin lightener","prod_type":""},{"id":421,"gen_cat":"Personal care","prod_fam":"skin protectant","prod_type":""},{"id":242,"gen_cat":"Personal care","prod_fam":"specialized bath products","prod_type":""},{"id":243,"gen_cat":"Personal care","prod_fam":"specialized bath products","prod_type":"bath oil"},{"id":244,"gen_cat":"Personal care","prod_fam":"specialized bath products","prod_type":"bath salts"},{"id":245,"gen_cat":"Personal care","prod_fam":"specialized bath products","prod_type":"bubble bath"},{"id":246,"gen_cat":"Personal care","prod_fam":"sunscreen","prod_type":""},{"id":360,"gen_cat":"Personal care","prod_fam":"topical skin treatment","prod_type":""},{"id":337,"gen_cat":"Personal care","prod_fam":"topical skin treatment","prod_type":"antibacterial treatment"},{"id":336,"gen_cat":"Personal care","prod_fam":"topical skin treatment","prod_type":"antifungal treatment"},{"id":420,"gen_cat":"Personal care","prod_fam":"topical skin treatment","prod_type":"antiseptic"},{"id":142,"gen_cat":"Personal care","prod_fam":"topical skin treatment","prod_type":"bite relief"},{"id":247,"gen_cat":"Pesticides","prod_fam":"","prod_type":""},{"id":248,"gen_cat":"Pesticides","prod_fam":"animal repellent","prod_type":""},{"id":249,"gen_cat":"Pesticides","prod_fam":"fungicide","prod_type":""},{"id":250,"gen_cat":"Pesticides","prod_fam":"insect repellent","prod_type":""},{"id":252,"gen_cat":"Pesticides","prod_fam":"insect repellent","prod_type":"insect repellent - skin"},{"id":253,"gen_cat":"Pesticides","prod_fam":"insecticide","prod_type":""},{"id":435,"gen_cat":"Pesticides","prod_fam":"Professional use pesticides","prod_type":""},{"id":254,"gen_cat":"Pesticides","prod_fam":"rodenticide","prod_type":""},{"id":255,"gen_cat":"Pet and animal care","prod_fam":"","prod_type":""},{"id":256,"gen_cat":"Pet and animal care","prod_fam":"all pets","prod_type":""},{"id":257,"gen_cat":"Pet and animal care","prod_fam":"all pets","prod_type":"other pet treatments"},{"id":258,"gen_cat":"Pet and animal care","prod_fam":"all pets","prod_type":"pesticide - pet"},{"id":259,"gen_cat":"Pet and animal care","prod_fam":"all pets","prod_type":"pet shampoo"},{"id":260,"gen_cat":"Pet and animal care","prod_fam":"all pets","prod_type":"pet stain cleaner"},{"id":357,"gen_cat":"Pet and animal care","prod_fam":"birds","prod_type":""},{"id":261,"gen_cat":"Pet and animal care","prod_fam":"cats","prod_type":""},{"id":262,"gen_cat":"Pet and animal care","prod_fam":"cats","prod_type":"cat litter"},{"id":432,"gen_cat":"Pet and animal care","prod_fam":"farm animals","prod_type":""},{"id":263,"gen_cat":"Pet and animal care","prod_fam":"fish","prod_type":""},{"id":264,"gen_cat":"Pet and animal care","prod_fam":"fish","prod_type":"aquarium"},{"id":265,"gen_cat":"Sports equipment","prod_fam":"","prod_type":""},{"id":266,"gen_cat":"Sports equipment","prod_fam":"bicycling","prod_type":""},{"id":267,"gen_cat":"Sports equipment","prod_fam":"bicycling","prod_type":"bicycle cleaner"},{"id":268,"gen_cat":"Sports equipment","prod_fam":"fishing","prod_type":""},{"id":269,"gen_cat":"Sports equipment","prod_fam":"fishing","prod_type":"reel oil"},{"id":458,"gen_cat":"Sports equipment","prod_fam":"hunting","prod_type":""},{"id":21,"gen_cat":"Sports equipment","prod_fam":"hunting","prod_type":"gun cleaner"},{"id":459,"gen_cat":"Sports equipment","prod_fam":"hunting","prod_type":"hunting scents"},{"id":270,"gen_cat":"Sports equipment","prod_fam":"skiing","prod_type":""},{"id":271,"gen_cat":"Sports equipment","prod_fam":"skiing","prod_type":"wax"},{"id":272,"gen_cat":"Vehicle","prod_fam":"","prod_type":""},{"id":273,"gen_cat":"Vehicle","prod_fam":"auto body work","prod_type":""},{"id":453,"gen_cat":"Vehicle","prod_fam":"auto body work","prod_type":"auto adhesives"},{"id":274,"gen_cat":"Vehicle","prod_fam":"auto body work","prod_type":"auto paint"},{"id":275,"gen_cat":"Vehicle","prod_fam":"auto body work","prod_type":"body repair"},{"id":276,"gen_cat":"Vehicle","prod_fam":"auto body work","prod_type":"detailing"},{"id":277,"gen_cat":"Vehicle","prod_fam":"auto body work","prod_type":"windows/windshield"},{"id":278,"gen_cat":"Vehicle","prod_fam":"boat care and maintenance","prod_type":""},{"id":280,"gen_cat":"Vehicle","prod_fam":"boat care and maintenance","prod_type":"boat cleaner"},{"id":282,"gen_cat":"Vehicle","prod_fam":"boat care and maintenance","prod_type":"boat engine fluids"},{"id":284,"gen_cat":"Vehicle","prod_fam":"car interior","prod_type":""},{"id":283,"gen_cat":"Vehicle","prod_fam":"car interior","prod_type":"auto air freshener"},{"id":285,"gen_cat":"Vehicle","prod_fam":"car surface treatment","prod_type":""},{"id":286,"gen_cat":"Vehicle","prod_fam":"car surface treatment","prod_type":"body cleaner"},{"id":287,"gen_cat":"Vehicle","prod_fam":"car surface treatment","prod_type":"body wax"},{"id":288,"gen_cat":"Vehicle","prod_fam":"car surface treatment","prod_type":"combination wash and wax"},{"id":289,"gen_cat":"Vehicle","prod_fam":"car surface treatment","prod_type":"degreaser"},{"id":290,"gen_cat":"Vehicle","prod_fam":"engine maintenance","prod_type":""},{"id":291,"gen_cat":"Vehicle","prod_fam":"engine maintenance","prod_type":"antifreeze"},{"id":292,"gen_cat":"Vehicle","prod_fam":"engine maintenance","prod_type":"auto fluids and additives"},{"id":293,"gen_cat":"Vehicle","prod_fam":"engine maintenance","prod_type":"auto lubricant"},{"id":294,"gen_cat":"Vehicle","prod_fam":"engine maintenance","prod_type":"auto refrigerant"},{"id":295,"gen_cat":"Vehicle","prod_fam":"engine maintenance","prod_type":"motor oil"},{"id":391,"gen_cat":"Vehicle","prod_fam":"recreational vehicle","prod_type":""},{"id":353,"gen_cat":"Vehicle","prod_fam":"recreational vehicle","prod_type":"holding tank treatment"},{"id":482,"gen_cat":"Agriculture","prod_fam":"","prod_type":""},{"id":483,"gen_cat":"Agriculture","prod_fam":"fertilizer","prod_type":""},{"id":485,"gen_cat":"Agriculture","prod_fam":"pesticides","prod_type":""},{"id":488,"gen_cat":"Agriculture","prod_fam":"pesticides","prod_type":"fungicide"},{"id":486,"gen_cat":"Agriculture","prod_fam":"pesticides","prod_type":"herbicide"},{"id":487,"gen_cat":"Agriculture","prod_fam":"pesticides","prod_type":"insecticide"},{"id":494,"gen_cat":"Agriculture","prod_fam":"pesticides","prod_type":"rodenticide"},{"id":484,"gen_cat":"Agriculture","prod_fam":"soil amendments","prod_type":""},{"id":319,"gen_cat":"Cleaning and safety","prod_fam":"","prod_type":""},{"id":363,"gen_cat":"Cleaning and safety","prod_fam":"cleaning products","prod_type":""},{"id":370,"gen_cat":"Cleaning and safety","prod_fam":"cleaning products","prod_type":"aircraft cleaners"},{"id":395,"gen_cat":"Cleaning and safety","prod_fam":"cleaning products","prod_type":"graffiti remover"},{"id":369,"gen_cat":"Cleaning and safety","prod_fam":"cleaning products","prod_type":"industrial carpet and floor cleaner"},{"id":367,"gen_cat":"Cleaning and safety","prod_fam":"cleaning products","prod_type":"industrial degreaser"},{"id":397,"gen_cat":"Cleaning and safety","prod_fam":"cleaning products","prod_type":"industrial deodorizer"},{"id":365,"gen_cat":"Cleaning and safety","prod_fam":"cleaning products","prod_type":"industrial dishwashing detergent"},{"id":413,"gen_cat":"Cleaning and safety","prod_fam":"cleaning products","prod_type":"industrial disinfectant"},{"id":416,"gen_cat":"Cleaning and safety","prod_fam":"cleaning products","prod_type":"industrial electrical cleaner"},{"id":464,"gen_cat":"Cleaning and safety","prod_fam":"cleaning products","prod_type":"industrial glass cleaners"},{"id":368,"gen_cat":"Cleaning and safety","prod_fam":"cleaning products","prod_type":"industrial hand cleaner"},{"id":414,"gen_cat":"Cleaning and safety","prod_fam":"cleaning products","prod_type":"industrial hand sanitizer"},{"id":366,"gen_cat":"Cleaning and safety","prod_fam":"cleaning products","prod_type":"industrial laundry detergent"},{"id":412,"gen_cat":"Cleaning and safety","prod_fam":"cleaning products","prod_type":"industrial surface cleaners"},{"id":465,"gen_cat":"Cleaning and safety","prod_fam":"cleaning products","prod_type":"restroom cleaner"},{"id":364,"gen_cat":"Cleaning and safety","prod_fam":"cleaning products","prod_type":"Sorbents and spill kits"},{"id":371,"gen_cat":"Cleaning and safety","prod_fam":"safety products","prod_type":""},{"id":373,"gen_cat":"Cleaning and safety","prod_fam":"safety products","prod_type":"emergency eye wash solution"},{"id":374,"gen_cat":"Cleaning and safety","prod_fam":"safety products","prod_type":"fire extinguishers"},{"id":372,"gen_cat":"Cleaning and safety","prod_fam":"safety products","prod_type":"personal protective equipment"},{"id":405,"gen_cat":"Cleaning and safety","prod_fam":"safety products","prod_type":"reflective materials"},{"id":472,"gen_cat":"Cleaning and safety","prod_fam":"safety products","prod_type":"safety lightsticks"},{"id":433,"gen_cat":"Construction","prod_fam":"","prod_type":""},{"id":446,"gen_cat":"Construction","prod_fam":"building construction","prod_type":""},{"id":406,"gen_cat":"Construction","prod_fam":"construction adhesives","prod_type":""},{"id":351,"gen_cat":"Construction","prod_fam":"marking paint","prod_type":""},{"id":434,"gen_cat":"Construction","prod_fam":"road construction","prod_type":""},{"id":318,"gen_cat":"Industrial products","prod_fam":"","prod_type":""},{"id":467,"gen_cat":"Industrial products","prod_fam":"adhesive removers","prod_type":""},{"id":398,"gen_cat":"Industrial products","prod_fam":"anti-seize","prod_type":""},{"id":452,"gen_cat":"Industrial products","prod_fam":"blast media","prod_type":""},{"id":382,"gen_cat":"Industrial products","prod_fam":"corrosion inhibitor","prod_type":""},{"id":401,"gen_cat":"Industrial products","prod_fam":"electrical insulation","prod_type":""},{"id":451,"gen_cat":"Industrial products","prod_fam":"food contact formulations","prod_type":""},{"id":381,"gen_cat":"Industrial products","prod_fam":"hydraulic fluid","prod_type":""},{"id":385,"gen_cat":"Industrial products","prod_fam":"hydraulic fluid","prod_type":"tractor hydraulic fluid"},{"id":466,"gen_cat":"Industrial products","prod_fam":"hydraulic fracturing fluid","prod_type":""},{"id":379,"gen_cat":"Industrial products","prod_fam":"industrial lubricants","prod_type":""},{"id":447,"gen_cat":"Industrial products","prod_fam":"labels","prod_type":""},{"id":358,"gen_cat":"Industrial products","prod_fam":"metalworking fluid","prod_type":""},{"id":380,"gen_cat":"Industrial products","prod_fam":"mold release agents","prod_type":""},{"id":475,"gen_cat":"Industrial products","prod_fam":"paint strippers","prod_type":""},{"id":386,"gen_cat":"Industrial products","prod_fam":"penetrating oil","prod_type":""},{"id":383,"gen_cat":"Industrial products","prod_fam":"refrigerant","prod_type":""},{"id":403,"gen_cat":"Industrial products","prod_fam":"vulcanizing fluid","prod_type":""},{"id":321,"gen_cat":"Laboratory supplies","prod_fam":"","prod_type":""},{"id":317,"gen_cat":"Medical/dental","prod_fam":"","prod_type":""},{"id":375,"gen_cat":"Medical/dental","prod_fam":"medical personal protective equipment","prod_type":""},{"id":301,"gen_cat":"Raw materials","prod_fam":"","prod_type":""},{"id":376,"gen_cat":"Raw materials","prod_fam":"adhesives","prod_type":""},{"id":355,"gen_cat":"Raw materials","prod_fam":"coatings","prod_type":""},{"id":410,"gen_cat":"Raw materials","prod_fam":"coatings","prod_type":"fireproof coatings"},{"id":492,"gen_cat":"Raw materials","prod_fam":"curing agents","prod_type":""},{"id":323,"gen_cat":"Raw materials","prod_fam":"flame retardants","prod_type":""},{"id":428,"gen_cat":"Raw materials","prod_fam":"food additive formulations","prod_type":""},{"id":470,"gen_cat":"Raw materials","prod_fam":"fragrance formulations","prod_type":""},{"id":384,"gen_cat":"Raw materials","prod_fam":"glass microspheres","prod_type":""},{"id":356,"gen_cat":"Raw materials","prod_fam":"inks","prod_type":""},{"id":400,"gen_cat":"Raw materials","prod_fam":"paints and primers","prod_type":""},{"id":422,"gen_cat":"Raw materials","prod_fam":"paints and primers","prod_type":"machinery paint"},{"id":477,"gen_cat":"Raw materials","prod_fam":"personal care formulations","prod_type":""},{"id":476,"gen_cat":"Raw materials","prod_fam":"pigments","prod_type":""},{"id":473,"gen_cat":"Raw materials","prod_fam":"resins","prod_type":""},{"id":394,"gen_cat":"Raw materials","prod_fam":"solvent formulations","prod_type":""},{"id":427,"gen_cat":"Raw materials","prod_fam":"surfactant formulations","prod_type":""},{"id":474,"gen_cat":"Raw materials","prod_fam":"thinners","prod_type":""},{"id":320,"gen_cat":"Specialty occupational products","prod_fam":"","prod_type":""},{"id":404,"gen_cat":"Specialty occupational products","prod_fam":"aviation","prod_type":""},{"id":408,"gen_cat":"Specialty occupational products","prod_fam":"aviation","prod_type":"aircraft coatings"},{"id":396,"gen_cat":"Specialty occupational products","prod_fam":"aviation","prod_type":"aircraft deicer"},{"id":378,"gen_cat":"Specialty occupational products","prod_fam":"aviation","prod_type":"aircraft fluids and additives"},{"id":407,"gen_cat":"Specialty occupational products","prod_fam":"dry cleaning chemicals","prod_type":""},{"id":324,"gen_cat":"Specialty occupational products","prod_fam":"firefighting agents","prod_type":""},{"id":399,"gen_cat":"Specialty occupational products","prod_fam":"gas leak detector","prod_type":""},{"id":387,"gen_cat":"Specialty occupational products","prod_fam":"law enforcement","prod_type":""},{"id":417,"gen_cat":"Specialty occupational products","prod_fam":"law enforcement","prod_type":"fingerprint detection"},{"id":418,"gen_cat":"Specialty occupational products","prod_fam":"law enforcement","prod_type":"fingerprinting ink"},{"id":332,"gen_cat":"Specialty occupational products","prod_fam":"law enforcement","prod_type":"pepper spray"},{"id":333,"gen_cat":"Specialty occupational products","prod_fam":"law enforcement","prod_type":"stun guns"},{"id":338,"gen_cat":"Specialty occupational products","prod_fam":"photography developing chemicals","prod_type":""},{"id":481,"gen_cat":"Specialty occupational products","prod_fam":"printing supplies","prod_type":""},{"id":377,"gen_cat":"Specialty occupational products","prod_fam":"water treatment chemicals","prod_type":""},{"id":116,"gen_cat":"Specialty occupational products","prod_fam":"welding","prod_type":""},{"id":448,"gen_cat":"Unknown or Indeterminate","prod_fam":"","prod_type":""}]''')
print(len(PUC_LIST), "PUCs loaded")
PUC_LIST[:3]


### Optional: re-fetch the live list instead of the embedded snapshot

If you'd rather pull the current live list (in case PUCs have been
added/removed/recategorized since this was captured), run this instead of
trusting the embedded `PUC_LIST` above -- it just re-does the same
extraction against the live page over plain `requests` (no browser needed
for this step, since the table's JSON is present in the server-rendered
HTML).

In [ ]:
import re, json, requests

def fetch_live_puc_list():
    resp = requests.get("https://comptox.epa.gov/chemexpo/pucs/", timeout=30)
    resp.raise_for_status()
    m = re.search(r'<script id="tabledata" type="application/json">(.*?)</script>', resp.text, re.S)
    if not m:
        raise RuntimeError("Couldn't find the tabledata script tag -- page structure may have changed.")
    data = json.loads(m.group(1))
    return [{"id": d["id"], "gen_cat": d["gen_cat"], "prod_fam": d["prod_fam"], "prod_type": d["prod_type"]} for d in data]

# Uncomment to use the live list instead of the embedded snapshot:
# PUC_LIST = fetch_live_puc_list()
# print(len(PUC_LIST), "PUCs fetched live")


## 3. Filename / folder helpers

In [ ]:
import re
from pathlib import Path

BASE_DIR = Path("/content/puc_downloads")

def sanitize(s):
    """Filesystem-safe segment: strip, then collapse anything not
    alnum into a single underscore."""
    s = (s or "").strip()
    if not s:
        return ""
    return re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")

def puc_filename(row):
    """PUC_<Gen Cat>_<Prod Fam>_<Prod Type>.xlsx, omitting any blank segment
    (about 1 in 4 PUCs have no Prod Type, and each Gen Cat has exactly one
    top-level row with no Prod Fam/Prod Type at all)."""
    parts = [sanitize(row["gen_cat"])]
    if row["prod_fam"]:
        parts.append(sanitize(row["prod_fam"]))
    if row["prod_type"]:
        parts.append(sanitize(row["prod_type"]))
    return "PUC_" + "_".join(parts) + ".xlsx"

def puc_folder(row):
    """One folder per Gen Cat, e.g. /content/puc_downloads/Batteries/"""
    folder = BASE_DIR / sanitize(row["gen_cat"])
    folder.mkdir(parents=True, exist_ok=True)
    return folder

# sanity check: confirms the 476 filenames are all distinct before
# downloading anything (they are, checked against the embedded snapshot --
# re-check here in case you switched to the live-fetched list above).
seen = {}
for row in PUC_LIST:
    fn = puc_filename(row)
    if fn in seen:
        print(f"WARNING: duplicate filename {fn!r} for PUC ids {seen[fn]} and {row['id']}")
    seen[fn] = row["id"]
print(f"{len(seen)} distinct filenames for {len(PUC_LIST)} PUCs")


## 4. Download loop

Visits each PUC detail page and clicks the "Download Products and Chemical
Weight Fractions" button, saving the result under its Gen Cat folder.
Resumable: a PUC whose target file already exists is skipped, so you can
re-run this cell after a crash or a Colab disconnect without starting
over. Failures (timeout, missing button, zero-product PUCs that may not
have a download button at all, etc.) are logged to `failed_pucs.csv`
rather than stopping the run.

A 1.5s pause between PUCs is included as a courtesy to the server across
476 sequential requests -- raise `REQUEST_DELAY_SECONDS` if you see
timeouts or errors that look like rate-limiting.

In [ ]:
import csv
import time
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError

REQUEST_DELAY_SECONDS = 1.5
DOWNLOAD_BUTTON_TEXT = "Download Products and Chemical Weight Fractions"
PAGE_LOAD_TIMEOUT_MS = 30000
DOWNLOAD_TIMEOUT_MS = 30000

failed = []

# Colab's own kernel already runs an asyncio event loop, which is why this
# uses Playwright's ASYNC API (await ...) rather than sync_playwright() --
# the sync API tries to start its own loop and Colab refuses that with
# "It looks like you are using Playwright Sync API inside the asyncio
# loop." Top-level `await` in a cell (as used below) is supported by
# Colab/Jupyter directly, no extra setup needed.

async def download_all():
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()

        for i, row in enumerate(PUC_LIST, start=1):
            dest = puc_folder(row) / puc_filename(row)
            if dest.exists():
                continue  # already downloaded on a previous run

            url = f"https://comptox.epa.gov/chemexpo/puc/{row['id']}/"
            print(f"[{i}/{len(PUC_LIST)}] {row['gen_cat']} / {row['prod_fam']} / {row['prod_type']} (id={row['id']})")
            try:
                await page.goto(url, wait_until="networkidle", timeout=PAGE_LOAD_TIMEOUT_MS)
                async with page.expect_download(timeout=DOWNLOAD_TIMEOUT_MS) as download_info:
                    # get_by_text finds the button by its visible label regardless
                    # of exact markup/class names -- more resilient to a CSS
                    # tweak on the site than a CSS-selector locator would be.
                    await page.get_by_text(DOWNLOAD_BUTTON_TEXT, exact=False).click()
                download = await download_info.value
                await download.save_as(dest)
            except PlaywrightTimeoutError as e:
                print(f"  TIMEOUT: {e}")
                failed.append({**row, "error": f"timeout: {e}"})
            except Exception as e:
                print(f"  FAILED: {e}")
                failed.append({**row, "error": str(e)})

            time.sleep(REQUEST_DELAY_SECONDS)

        await browser.close()

await download_all()

print(f"\nDone. {len(PUC_LIST) - len(failed)} succeeded (or already present), {len(failed)} failed.")

if failed:
    with open("/content/failed_pucs.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["id", "gen_cat", "prod_fam", "prod_type", "error"])
        w.writeheader()
        w.writerows(failed)
    print("Failures logged to /content/failed_pucs.csv -- re-run this cell after fixing the")
    print("cause (e.g. the button locator) to retry just the missing ones.")


## 5. Verify what was downloaded

In [ ]:
total_files = sum(1 for _ in BASE_DIR.rglob("*.xlsx"))
print(f"{total_files} files across {len(list(BASE_DIR.iterdir()))} Gen Cat folders under {BASE_DIR}")
for folder in sorted(BASE_DIR.iterdir()):
    n = sum(1 for _ in folder.glob("*.xlsx"))
    print(f"  {folder.name}: {n}")


## 6. Upload to Google Drive

Authenticates as you (a sign-in popup will appear), then mirrors the local
`Gen Cat -> PUC_....xlsx` folder tree into the target Drive folder,
creating each Gen Cat subfolder if it doesn't already exist there. Also
resumable: a file already present (by name) in its target Drive folder is
skipped rather than re-uploaded, so re-running after an interruption picks
up where it left off.

In [ ]:
DRIVE_FOLDER_ID = "1tDncfoHC14dWet34H34SEukt1BLi4aPT"  # from the shared folder's URL; replace if different

from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

drive_service = build("drive", "v3")

def find_child_folder(parent_id, name):
    q = (
        f"'{parent_id}' in parents and name = '{name}' "
        "and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    )
    resp = drive_service.files().list(q=q, fields="files(id, name)").execute()
    files = resp.get("files", [])
    return files[0]["id"] if files else None

def get_or_create_child_folder(parent_id, name):
    existing = find_child_folder(parent_id, name)
    if existing:
        return existing
    metadata = {"name": name, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_id]}
    created = drive_service.files().create(body=metadata, fields="id").execute()
    return created["id"]

def file_exists_in_folder(parent_id, filename):
    q = f"'{parent_id}' in parents and name = '{filename}' and trashed = false"
    resp = drive_service.files().list(q=q, fields="files(id)").execute()
    return len(resp.get("files", [])) > 0

uploaded, skipped, upload_failed = 0, 0, []
gen_cat_folders = sorted(p for p in BASE_DIR.iterdir() if p.is_dir())
for local_folder in gen_cat_folders:
    drive_folder_id = get_or_create_child_folder(DRIVE_FOLDER_ID, local_folder.name)
    for local_file in sorted(local_folder.glob("*.xlsx")):
        if file_exists_in_folder(drive_folder_id, local_file.name):
            skipped += 1
            continue
        try:
            media = MediaFileUpload(str(local_file))
            drive_service.files().create(
                body={"name": local_file.name, "parents": [drive_folder_id]},
                media_body=media,
                fields="id",
            ).execute()
            uploaded += 1
        except Exception as e:
            print(f"  FAILED to upload {local_file}: {e}")
            upload_failed.append(str(local_file))

print(f"\nUploaded {uploaded}, skipped {skipped} (already present), {len(upload_failed)} failed.")
if upload_failed:
    print("Re-run this cell to retry the failed ones.")
